# Model Training

In [ ]:
# %pip install xgboost

## Setup, Imports, and MLflow Initialization

In [1]:
# ---------------------------------------------------------
# Section 1.1: Setup & Imports (Sandboxed MLflow Setup)
# ---------------------------------------------------------
import pandas as pd
import numpy as np
import os
import pathlib
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn & XGBoost
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score, f1_score
import xgboost as xgb

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate, Dropout, Masking
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

# MLflow
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.keras

# --- MLflow Setup: The Quarantine Zone ---

# 1. Define and create the dedicated mlflow/ directory at the project root
mlflow_dir = pathlib.Path("../mlflow").resolve()
mlflow_dir.mkdir(parents=True, exist_ok=True)

# 2. Define and create the artifacts/ folder inside it
artifact_dir = mlflow_dir / "artifacts"
artifact_dir.mkdir(exist_ok=True)
artifact_uri = artifact_dir.as_uri()

# 3. Point the tracking database to sit directly inside the mlflow/ folder
mlflow.set_tracking_uri("sqlite:///../mlflow/mlruns.db")

# 4. Use a clean experiment name
experiment_name = "bluecradle_training"

# 5. Create the experiment explicitly pointing to our sandboxed artifact folder
try:
    mlflow.create_experiment(
        name=experiment_name, 
        artifact_location=artifact_uri
    )
except mlflow.exceptions.MlflowException:
    pass

# 6. Set the active experiment
mlflow.set_experiment(experiment_name)

print("All libraries imported successfully.")
print(f"MLflow DB locked to: {mlflow.get_tracking_uri()}")
print(f"Artifacts locked to: {artifact_uri}")

All libraries imported successfully.
MLflow DB locked to: sqlite:///../mlflow/mlruns.db
Artifacts locked to: file:///D:/final-year-project-team_bluecradle-ml-system/mlflow/artifacts


## Load Prepared Data

In [2]:
# ---------------------------------------------------------
# Section 1.2: Load Prepared Data
# ---------------------------------------------------------
data_dir = "../data/prepared_data"

# --- 1. DHS Data (Shallow Learning) ---
X_dhs_train = pd.read_csv(f"{data_dir}/X_dhs_train.csv")
y_dhs_train = pd.read_csv(f"{data_dir}/y_dhs_train.csv")['label'] # Extracted as Series

X_dhs_test = pd.read_csv(f"{data_dir}/X_dhs_test.csv")
y_dhs_test = pd.read_csv(f"{data_dir}/y_dhs_test.csv")['label']

print("--- DHS Data Loaded ---")
print(f"X_dhs_train: {X_dhs_train.shape} | y_dhs_train: {y_dhs_train.shape}")
print(f"X_dhs_test: {X_dhs_test.shape}  | y_dhs_test: {y_dhs_test.shape}")

# --- 2. MAL-ED Tensors (Deep Learning) ---
maled_data = np.load(f"{data_dir}/maled_tensors.npz", allow_pickle=True)

X_train_seq = maled_data['X_train_seq']
X_train_static = maled_data['X_train_static']
y_train_seq = maled_data['y_train_seq']

X_test_seq = maled_data['X_test_seq']
X_test_static = maled_data['X_test_static']
y_test_seq = maled_data['y_test_seq']

# Extract class weights (loaded as a 0-d numpy object, .item() extracts the native dictionary)
maled_class_weights = maled_data['class_weights'].item()

print("\n--- MAL-ED Data Loaded ---")
print(f"X_train_seq: {X_train_seq.shape} | X_train_static: {X_train_static.shape} | y_train_seq: {y_train_seq.shape}")
print(f"X_test_seq: {X_test_seq.shape}  | X_test_static: {X_test_static.shape}  | y_test_seq: {y_test_seq.shape}")
print(f"Class Weights: {maled_class_weights}")

--- DHS Data Loaded ---
X_dhs_train: (9240, 18) | y_dhs_train: (9240,)
X_dhs_test: (999, 18)  | y_dhs_test: (999,)

--- MAL-ED Data Loaded ---
X_train_seq: (560, 6, 8) | X_train_static: (560, 3) | y_train_seq: (560, 6)
X_test_seq: (140, 6, 8)  | X_test_static: (140, 3)  | y_test_seq: (140, 6)
Class Weights: {np.int64(0): np.float64(0.599057714958775), np.int64(1): np.float64(1.6953333333333334), np.int64(2): np.float64(1.349787685774947)}


## Random Forest (Baseline)

In [3]:
# ---------------------------------------------------------
# Section 2: Random Forest (Baseline)
# ---------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# Start MLflow run
with mlflow.start_run(run_name="random_forest_baseline"):
    
    # 1. Define hyperparameters
    rf_params = {
        "n_estimators": 100,
        "max_depth": 10,
        "class_weight": "balanced",
        "random_state": 42
    }
    
    # Log hyperparameters to MLflow
    mlflow.log_params(rf_params)

    # 2. Initialize and Train Model
    print("Training Random Forest...")
    rf_model = RandomForestClassifier(**rf_params)
    rf_model.fit(X_dhs_train, y_dhs_train)

    # 3. Predict on DHS Test Set
    y_pred_rf = rf_model.predict(X_dhs_test)

    # 4. Calculate Metrics (Wrapped in float() to satisfy MLflow)
    acc = float(accuracy_score(y_dhs_test, y_pred_rf))
    f1 = float(f1_score(y_dhs_test, y_pred_rf, average='weighted'))
    
    report = classification_report(y_dhs_test, y_pred_rf, output_dict=True)
    
    # Safely extract the nested dictionaries to satisfy Pylance
    sam_metrics = report.get('2', {})
    mam_metrics = report.get('1', {})

    # Extract recall and cast to standard Python float
    sam_recall = float(sam_metrics.get('recall', 0.0)) if isinstance(sam_metrics, dict) else 0.0
    mam_recall = float(mam_metrics.get('recall', 0.0)) if isinstance(mam_metrics, dict) else 0.0

    # Log Metrics to MLflow
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_weighted", f1)
    mlflow.log_metric("sam_recall", sam_recall)
    mlflow.log_metric("mam_recall", mam_recall)

    print(f"RF SAM Recall: {sam_recall:.4f} | MAM Recall: {mam_recall:.4f} | Accuracy: {acc:.4f}")

    # 5. Generate and Log Confusion Matrix Artifact
    fig_cm, ax_cm = plt.subplots(figsize=(6, 4))
    sns.heatmap(confusion_matrix(y_dhs_test, y_pred_rf), annot=True, fmt='d', cmap='Blues', ax=ax_cm)
    ax_cm.set_title("Random Forest Confusion Matrix")
    ax_cm.set_xlabel("Predicted (0=Normal, 1=MAM, 2=SAM)")
    ax_cm.set_ylabel("Actual")
    fig_cm.savefig("rf_confusion_matrix.png")
    mlflow.log_artifact("rf_confusion_matrix.png")
    plt.close(fig_cm)

    # 6. Generate and Log Feature Importances Artifact
    fig_fi, ax_fi = plt.subplots(figsize=(8, 6))
    importances = pd.Series(rf_model.feature_importances_, index=X_dhs_train.columns).sort_values(ascending=True)
    importances.tail(10).plot(kind='barh', ax=ax_fi)
    ax_fi.set_title("Top 10 Feature Importances - Random Forest")
    fig_fi.savefig("rf_feature_importance.png")
    mlflow.log_artifact("rf_feature_importance.png")
    plt.close(fig_fi)

    # 7. Log the Model itself
    mlflow.sklearn.log_model(rf_model, "random_forest_model")
    
    print("✅ Run 'random_forest_baseline' completed and logged to MLflow.")

Training Random Forest...
RF SAM Recall: 0.9930 | MAM Recall: 1.0000 | Accuracy: 0.9930


2026/05/26 14:08:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/26 14:08:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Run 'random_forest_baseline' completed and logged to MLflow.


## XGBoost

In [4]:
# ---------------------------------------------------------
# Section 3: XGBoost
# ---------------------------------------------------------

# Start MLflow run
with mlflow.start_run(run_name="xgboost"):
    
    # 1. Define hyperparameters
    xgb_params = {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "objective": "multi:softmax",
        "num_class": 3,
        "random_state": 42
    }
    
    # Log hyperparameters to MLflow
    mlflow.log_params(xgb_params)

    # 2. Initialize and Train Model
    print("Training XGBoost...")
    # XGBoost requires target labels to be strictly sequentially encoded starting from 0.
    # Our labels are exactly 0, 1, 2, which is perfect.
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(X_dhs_train, y_dhs_train)

    # 3. Predict on DHS Test Set
    y_pred_xgb = xgb_model.predict(X_dhs_test)

    # 4. Calculate Metrics (Wrapped in float() to satisfy MLflow)
    acc = float(accuracy_score(y_dhs_test, y_pred_xgb))
    f1 = float(f1_score(y_dhs_test, y_pred_xgb, average='weighted'))
    
    report = classification_report(y_dhs_test, y_pred_xgb, output_dict=True)
    
    sam_metrics = report.get('2', {})
    mam_metrics = report.get('1', {})

    sam_recall = float(sam_metrics.get('recall', 0.0)) if isinstance(sam_metrics, dict) else 0.0
    mam_recall = float(mam_metrics.get('recall', 0.0)) if isinstance(mam_metrics, dict) else 0.0

    # Log Metrics to MLflow
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_weighted", f1)
    mlflow.log_metric("sam_recall", sam_recall)
    mlflow.log_metric("mam_recall", mam_recall)

    print(f"XGB SAM Recall: {sam_recall:.4f} | MAM Recall: {mam_recall:.4f} | Accuracy: {acc:.4f}")

    # 5. Generate and Log Confusion Matrix Artifact
    fig_cm, ax_cm = plt.subplots(figsize=(6, 4))
    sns.heatmap(confusion_matrix(y_dhs_test, y_pred_xgb), annot=True, fmt='d', cmap='Blues', ax=ax_cm)
    ax_cm.set_title("XGBoost Confusion Matrix")
    ax_cm.set_xlabel("Predicted (0=Normal, 1=MAM, 2=SAM)")
    ax_cm.set_ylabel("Actual")
    fig_cm.savefig("xgb_confusion_matrix.png")
    mlflow.log_artifact("xgb_confusion_matrix.png")
    plt.close(fig_cm)

    # 6. Generate and Log Feature Importances Artifact
    fig_fi, ax_fi = plt.subplots(figsize=(8, 6))
    importances = pd.Series(xgb_model.feature_importances_, index=X_dhs_train.columns).sort_values(ascending=True)
    importances.tail(10).plot(kind='barh', ax=ax_fi)
    ax_fi.set_title("Top 10 Feature Importances - XGBoost")
    fig_fi.savefig("xgb_feature_importance.png")
    mlflow.log_artifact("xgb_feature_importance.png")
    plt.close(fig_fi)

    # 7. Log the Model itself
    mlflow.xgboost.log_model(xgb_model, "xgboost_model")
    
    print("✅ Run 'xgboost' completed and logged to MLflow.")

Training XGBoost...


2026/05/26 14:08:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGB SAM Recall: 1.0000 | MAM Recall: 1.0000 | Accuracy: 1.0000
✅ Run 'xgboost' completed and logged to MLflow.


## LSTM + Feedforward Hybrid

In [5]:
# ---------------------------------------------------------
# Section 4: LSTM + Feedforward Hybrid
# ---------------------------------------------------------
import matplotlib.pyplot as plt
from tensorflow.keras.layers import RepeatVector
from tensorflow.keras.optimizers import Adam
import os

# 1. Pre-process Targets and Weights
# Keras cannot one-hot encode negative numbers. We temporarily map padding (-1) to 0.
y_train_safe = np.where(y_train_seq == -1, 0, y_train_seq)
y_test_safe = np.where(y_test_seq == -1, 0, y_test_seq)

y_train_cat = to_categorical(y_train_safe, num_classes=3)
y_test_cat = to_categorical(y_test_safe, num_classes=3)

# Build a 2D sample_weights array to dynamically handle class imbalance AND padding
train_sample_weights = np.zeros_like(y_train_seq, dtype='float32')
for class_id, weight in maled_class_weights.items():
    train_sample_weights[y_train_seq == class_id] = float(weight)

# Test weights: Give a weight of 1.0 to valid data, leave padding as 0.0
test_sample_weights = np.zeros_like(y_test_seq, dtype='float32')
test_sample_weights[y_test_seq != -1] = 1.0 

# Start MLflow run
with mlflow.start_run(run_name="lstm_feedforward_hybrid"):
    
    # 2. Define Hyperparameters
    nn_params = {
        "lstm_units": 64,
        "static_dense_units": 16,
        "combined_dense_units": 32,
        "dropout_rate": 0.3,
        "learning_rate": 0.001,
        "batch_size": 32,
        "epochs": 50
    }
    mlflow.log_params(nn_params)

    # 3. Build Architecture
    # --- Sequential Branch ---
    seq_input = Input(shape=(X_train_seq.shape[1], X_train_seq.shape[2]), name="seq_input")
    masked_seq = Masking(mask_value=-999.0)(seq_input)
    lstm_out = LSTM(nn_params["lstm_units"], return_sequences=True, dropout=nn_params["dropout_rate"])(masked_seq)
    
    # --- Static Branch ---
    static_input = Input(shape=(X_train_static.shape[1],), name="static_input")
    static_dense = Dense(nn_params["static_dense_units"], activation='relu')(static_input)
    
    # Repeat static vector to match the 6 sequence timesteps
    repeated_static = RepeatVector(X_train_seq.shape[1])(static_dense)
    
    # --- Concatenation & Output ---
    concat = Concatenate()([lstm_out, repeated_static])
    dense1 = Dense(nn_params["combined_dense_units"], activation='relu')(concat)
    dropout1 = Dropout(nn_params["dropout_rate"])(dense1)
    output = Dense(3, activation='softmax', name="output")(dropout1)
    
    model = Model(inputs=[seq_input, static_input], outputs=output)
    
    # 4. Compile Model
    optimizer = Adam(learning_rate=nn_params["learning_rate"])
    # Keras 3 automatically handles temporal weights based on the array shape provided in fit()
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    
    # 5. Callbacks
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    checkpoint_path = "temp_lstm_model.keras"
    checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True)
    
    # 6. Train Model
    print("Training LSTM Hybrid Model...")
    history = model.fit(
        [X_train_seq, X_train_static], y_train_cat,
        sample_weight=train_sample_weights,
        validation_data=([X_test_seq, X_test_static], y_test_cat, test_sample_weights),
        epochs=nn_params["epochs"],
        batch_size=nn_params["batch_size"],
        callbacks=[early_stop, checkpoint],
        verbose=1
    )
    
    # 7. Evaluate and Flatten Predictions for Clinical Metrics
    y_pred_probs = model.predict([X_test_seq, X_test_static])
    y_pred_classes = np.argmax(y_pred_probs, axis=-1)
    
    # We strictly filter out the -1 padding to calculate true clinical recall
    valid_mask = y_test_seq != -1
    y_true_flat = y_test_seq[valid_mask]
    y_pred_flat = y_pred_classes[valid_mask]
    
    acc = float(accuracy_score(y_true_flat, y_pred_flat))
    f1 = float(f1_score(y_true_flat, y_pred_flat, average='weighted'))
    
    report = classification_report(y_true_flat, y_pred_flat, output_dict=True)
    
    # Safely extract metrics
    sam_metrics = report.get('2', {})
    mam_metrics = report.get('1', {})
    sam_recall = float(sam_metrics.get('recall', 0.0)) if isinstance(sam_metrics, dict) else 0.0
    mam_recall = float(mam_metrics.get('recall', 0.0)) if isinstance(mam_metrics, dict) else 0.0
    
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_weighted", f1)
    mlflow.log_metric("sam_recall", sam_recall)
    mlflow.log_metric("mam_recall", mam_recall)
    
    print(f"\nLSTM SAM Recall: {sam_recall:.4f} | MAM Recall: {mam_recall:.4f} | Accuracy: {acc:.4f}")
    
    # 8. Generate and Log Training History Artifact
    fig_hist, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax_loss.plot(history.history['loss'], label='Train Loss')
    ax_loss.plot(history.history['val_loss'], label='Val Loss')
    ax_loss.set_title('Model Loss')
    ax_loss.legend()
    
    ax_acc.plot(history.history['accuracy'], label='Train Acc')
    ax_acc.plot(history.history['val_accuracy'], label='Val Acc')
    ax_acc.set_title('Model Accuracy')
    ax_acc.legend()
    
    fig_hist.savefig("lstm_training_history.png")
    mlflow.log_artifact("lstm_training_history.png")
    plt.close(fig_hist)
    
    # 9. Log the Model 
    # Load the best weights from the checkpoint before sending it to MLflow
    if os.path.exists(checkpoint_path):
        model.load_weights(checkpoint_path)
        mlflow.keras.log_model(model, "lstm_hybrid_model")
        os.remove(checkpoint_path) # Clean up the local temp file
        
    print("✅ Run 'lstm_feedforward_hybrid' completed and logged to MLflow.")

Training LSTM Hybrid Model...
Epoch 1/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.4574 - loss: 0.7963 - val_accuracy: 0.6202 - val_loss: 0.7261
Epoch 2/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5994 - loss: 0.7195 - val_accuracy: 0.6917 - val_loss: 0.6156
Epoch 3/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6741 - loss: 0.6310 - val_accuracy: 0.7310 - val_loss: 0.4925
Epoch 4/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6911 - loss: 0.5691 - val_accuracy: 0.7405 - val_loss: 0.4051
Epoch 5/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6985 - loss: 0.5293 - val_accuracy: 0.7464 - val_loss: 0.3743
Epoch 6/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7292 - loss: 0.4792 - val_accuracy: 0.7583 - val_loss: 0.3377
Epoch 7/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7131 - loss: 0.4698 - val_accuracy: 0.7464 - val_loss: 0.3301
Epoch 8/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7241 - loss: 0.4599 -

2026/05/26 14:09:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



LSTM SAM Recall: 0.8249 | MAM Recall: 0.9576 | Accuracy: 0.8938


2026/05/26 14:09:00 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


✅ Run 'lstm_feedforward_hybrid' completed and logged to MLflow.


## Quick Comparison Summary

In [6]:
# ---------------------------------------------------------
# Section 5: Quick Comparison Summary
# ---------------------------------------------------------
import mlflow
import pandas as pd
from IPython.display import display

# 1. Get the current active experiment
experiment = mlflow.get_experiment_by_name("bluecradle_training")

# 2. Programmatically pull all runs into a Pandas DataFrame
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# 3. Filter down to just the metrics we care about
# MLflow automatically prepends 'tags.', 'metrics.', and 'params.' to column names
cols_to_keep = [
    'tags.mlflow.runName', 
    'metrics.sam_recall', 
    'metrics.mam_recall', 
    'metrics.accuracy', 
    'metrics.f1_weighted'
]

# Ensure we only try to pull columns that actually exist (in case a run failed midway)
existing_cols = [col for col in cols_to_keep if col in runs_df.columns]
summary_df = runs_df[existing_cols].copy()

# 4. Clean up the column names for a professional display
summary_df.rename(columns={
    'tags.mlflow.runName': 'Model Architecture',
    'metrics.sam_recall': 'SAM Recall',
    'metrics.mam_recall': 'MAM Recall',
    'metrics.accuracy': 'Overall Accuracy',
    'metrics.f1_weighted': 'Weighted F1'
}, inplace=True)

# 5. Sort by our primary clinical metric: SAM Recall (descending)
summary_df = summary_df.sort_values(by='SAM Recall', ascending=False).reset_index(drop=True)

# 6. Display the final leaderboard
print("--- BlueCradle Model Leaderboard ---")
display(summary_df)

--- BlueCradle Model Leaderboard ---


,Model Architecture,SAM Recall,MAM Recall,Overall Accuracy,Weighted F1
0,xgboost,1.000000,1.000000,1.000000,1.000000
1,xgboost,1.000000,1.000000,1.000000,1.000000
2,random_forest_baseline,0.992958,1.000000,0.992993,0.993094
3,random_forest_baseline,0.992958,1.000000,0.992993,0.993094
4,lstm_feedforward_hybrid,0.824859,0.957627,0.893819,0.900698
5,lstm_feedforward_hybrid,0.745763,0.966102,0.870048,0.879043


## Conclusion & Next Steps

All three candidate models (Random Forest, XGBoost, and the LSTM Hybrid) have been successfully trained and logged locally using MLflow. 

While the table above provides a quick glance at our primary clinical metrics (specifically prioritizing **SAM Recall** and **MAM Recall** to minimize false negatives in a healthcare context), a deeper architectural evaluation is required.

**Next Phase:** In `3_model_evaluation.ipynb`, we will load these trained artifacts from the MLflow registry to conduct rigorous threshold tuning, generate precision-recall curves, interpret feature importance (using SHAP), and make the final selection for the BlueCradle backend API.